<a href="https://colab.research.google.com/github/Faisaleka21/Machine_Learning/blob/main/K_Means_artikel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

untuk kode dari pemilihan filter berikut ini :

In [ ]:
# ============================================================================
# IMPORT LIBRARY
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# TAHAP 1. MEMUAT DATASET
# ============================================================================
print("=" * 80)
print("IMPLEMENTASI K-MEANS CLUSTERING PADA DATASET PEMESANAN OPAK GAMBIR")
print("FITUR CLUSTERING: JUMLAH & HARGA")
print("=" * 80)

# Membaca dataset
df = pd.read_csv("https://raw.githubusercontent.com/Faisaleka21/Machine_Learning/refs/heads/main/data_set/kresnofarmok.csv",sep=';')
print(f"\nDataset berhasil dimuat.")
print(f"Jumlah data: {len(df)} baris")
print(f"Jumlah atribut: {len(df.columns)} kolom")
print(f"Atribut: {list(df.columns)}")


In [ ]:
# ============================================================================
# TAHAP 2. MENAMPILKAN INFORMASI DATASET DAN STATISTIK DESKRIPTIF
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 2. INFORMASI DATASET DAN STATISTIK DESKRIPTIF")
print("=" * 80)

# Tampilkan 10 data pertama
print("\n10 Data Pertama:")
print(df.head(10).to_string())

# Informasi dataset
print(f"\nInformasi Dataset:")
print(f"  Jumlah baris : {df.shape[0]}")
print(f"  Jumlah kolom : {df.shape[1]}")

# Statistik deskriptif untuk jumlah dan harga
print(f"\nStatistik Deskriptif Atribut 'jumlah' dan 'harga':")
print(df[['jumlah', 'harga']].describe().to_string())

# Cek missing value
print(f"\nMissing Value:")
print(f"  jumlah: {df['jumlah'].isnull().sum()} missing")
print(f"  harga : {df['harga'].isnull().sum()} missing")

# Informasi tipe data
print(f"\nTipe Data:")
print(f"  jumlah: {df['jumlah'].dtype}")
print(f"  harga : {df['harga'].dtype}")

In [ ]:
# ============================================================================
# TAHAP 3. MENANGANI MISSING VALUE
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 3. MENANGANI MISSING VALUE")
print("=" * 80)

# Konversi ke numerik dan tangani missing value
for col in ['jumlah', 'harga']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    missing_before = df[col].isnull().sum()
    if missing_before > 0:
        # Isi dengan median (lebih robust terhadap outlier)
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  {col}: {missing_before} missing value diisi dengan median ({median_val})")
    else:
        print(f"  {col}: Tidak ada missing value")

# Verifikasi tidak ada missing value tersisa
print(f"\nMissing value setelah penanganan: {df[['jumlah', 'harga']].isnull().sum().sum()}")

In [ ]:
# ============================================================================
# TAHAP 4. EKSTRAKSI FITUR DAN NORMALISASI DATA
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 4. EKSTRAKSI FITUR DAN NORMALISASI DATA (MinMaxScaler)")
print("=" * 80)

# Ekstraksi fitur jumlah dan harga
X = df[['jumlah', 'harga']].values

print(f"Dimensi data sebelum normalisasi: {X.shape}")
print(f"Range jumlah sebelum normalisasi: {X[:, 0].min():.0f} - {X[:, 0].max():.0f}")
print(f"Range harga sebelum normalisasi : {X[:, 1].min():.0f} - {X[:, 1].max():.0f}")

# Normalisasi menggunakan MinMaxScaler
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

print(f"\nRange jumlah setelah normalisasi: {X_normalized[:, 0].min():.4f} - {X_normalized[:, 0].max():.4f}")
print(f"Range harga setelah normalisasi : {X_normalized[:, 1].min():.4f} - {X_normalized[:, 1].max():.4f}")

# Tampilkan 5 data pertama setelah normalisasi
print(f"\n5 data pertama setelah normalisasi:")
for i in range(min(5, len(X_normalized))):
    print(f"  Data {i+1}: jumlah={X_normalized[i, 0]:.4f}, harga={X_normalized[i, 1]:.4f}")

In [ ]:
# ============================================================================
# TAHAP 5. MENENTUKAN JUMLAH CLUSTER OPTIMAL (ELBOW & SILHOUETTE)
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 5. MENENTUKAN JUMLAH CLUSTER OPTIMAL")
print("=" * 80)
print("Metode: Elbow Method & Silhouette Score")

# Uji K dari 2 sampai 10
K_range = range(2, 11)
inertia_values = []
silhouette_values = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_normalized)
    inertia_values.append(kmeans.inertia_)
    silhouette_values.append(silhouette_score(X_normalized, labels))
    print(f"  K={k}: Inertia={kmeans.inertia_:.4f}, Silhouette Score={silhouette_values[-1]:.4f}")

# Visualisasi Elbow Method dan Silhouette Score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Elbow
axes[0].plot(K_range, inertia_values, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Jumlah Cluster (K)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12, fontweight='bold')
axes[0].set_title('Elbow Method - Menentukan K Optimal', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
# Tandai titik optimal (misal K=3)
axes[0].axvline(x=3, color='red', linestyle='--', linewidth=2, label='K=3 (Elbow Point)')
axes[0].legend()

# Plot Silhouette Score
axes[1].plot(K_range, silhouette_values, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Jumlah Cluster (K)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Silhouette Score - Menentukan K Optimal', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
# Tandai K dengan silhouette tertinggi
best_k_sil = K_range[np.argmax(silhouette_values)]
best_sil = max(silhouette_values)
axes[1].axvline(x=best_k_sil, color='red', linestyle='--', linewidth=2,
                label=f'K={best_k_sil} (Best: {best_sil:.4f})')
axes[1].legend()

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

# Tentukan K optimal
best_k = 3  # Sesuai kebutuhan penelitian: Tinggi, Sedang, Rendah
print(f"\n>>> JUMLAH CLUSTER OPTIMAL: K = {best_k} <<<")
print(f"Alasan pemilihan K={best_k}:")
print(f"  1. Sesuai tujuan penelitian: mengelompokkan menjadi 3 kategori")
print(f"     (Pemesanan Tinggi, Sedang, Rendah)")
if best_k == best_k_sil:
    print(f"  2. Silhouette Score tertinggi pada K={best_k}")
print(f"  3. Terlihat elbow point pada K={best_k} di Elbow Method")

In [ ]:
# ============================================================================
# TAHAP 6. MENERAPKAN ALGORITMA K-MEANS (K=3)
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 6. MENERAPKAN ALGORITMA K-MEANS (K=3)")
print("=" * 80)

# Terapkan K-Means dengan K=3
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_normalized)

print("K-Means clustering berhasil diterapkan.")
print(f"Jumlah iterasi: {kmeans_final.n_iter_}")

In [ ]:
# ============================================================================
# TAHAP 7. MENAMPILKAN NILAI CENTROID SETIAP CLUSTER
# ============================================================================
import pandas as pd

# ============================================================================
# TAHAP 7. MENAMPILKAN NILAI CENTROID SETIAP CLUSTER (DALAM TABEL)
# ============================================================================

print("\n" + "=" * 80)
print("TAHAP 7. NILAI CENTROID SETIAP CLUSTER")
print("=" * 80)

# =========================
# Centroid normalisasi
# =========================
centroids_normalized = kmeans_final.cluster_centers_

df_centroid_norm = pd.DataFrame(
    centroids_normalized,
    columns=["jumlah", "harga"]
)
df_centroid_norm.index = [f"Cluster {i}" for i in range(len(df_centroid_norm))]

print("\nCentroid (skala normalisasi 0-1):")
print(df_centroid_norm)

# =========================
# Centroid skala asli
# =========================
centroids_original = scaler.inverse_transform(centroids_normalized)

df_centroid_orig = pd.DataFrame(
    centroids_original,
    columns=["jumlah (pcs)", "harga (Rp)"]
)
df_centroid_orig.index = [f"Cluster {i}" for i in range(len(df_centroid_orig))]

# format biar lebih rapi (opsional)
df_centroid_orig["jumlah (pcs)"] = df_centroid_orig["jumlah (pcs)"].round(0).astype(int)
df_centroid_orig["harga (Rp)"] = df_centroid_orig["harga (Rp)"].round(0).astype(int)

print("\nCentroid (skala asli):")
print(df_centroid_orig)

In [ ]:
# ============================================================================
# TAHAP 8. JUMLAH ANGGOTA SETIAP CLUSTER
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 8. JUMLAH ANGGOTA SETIAP CLUSTER")
print("=" * 80)

distribusi = df['cluster'].value_counts().sort_index()
print(f"\n{'Cluster':<10} {'Jumlah Anggota':<18} {'Persentase':<12}")
print("-" * 40)
for cluster_id in sorted(distribusi.index):
    jumlah_anggota = distribusi[cluster_id]
    persentase = (jumlah_anggota / len(df)) * 100
    print(f"Cluster {cluster_id:<3} {jumlah_anggota:<18} {persentase:.1f}%")
print("-" * 40)
print(f"{'Total':<10} {len(df):<18} 100.0%")

In [ ]:
# ============================================================================
# TAHAP 9. ANALISIS KARAKTERISTIK CLUSTER
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 9. ANALISIS KARAKTERISTIK CLUSTER")
print("=" * 80)

# Hitung statistik per cluster
cluster_stats = {}
for cluster_id in sorted(df['cluster'].unique()):
    data_cluster = df[df['cluster'] == cluster_id]
    cluster_stats[cluster_id] = {
        'count': len(data_cluster),
        'mean_jumlah': data_cluster['jumlah'].mean(),
        'median_jumlah': data_cluster['jumlah'].median(),
        'min_jumlah': data_cluster['jumlah'].min(),
        'max_jumlah': data_cluster['jumlah'].max(),
        'std_jumlah': data_cluster['jumlah'].std(),
        'mean_harga': data_cluster['harga'].mean(),
        'min_harga': data_cluster['harga'].min(),
        'max_harga': data_cluster['harga'].max()
    }

# Tampilkan statistik per cluster
print(f"\n{'Cluster':<10} {'Count':<8} {'Mean Jml':<12} {'Min Jml':<10} {'Max Jml':<10} {'Mean Harga':<15} {'Min Harga':<15} {'Max Harga':<15}")
print("-" * 95)
for cluster_id in sorted(df['cluster'].unique()):
    s = cluster_stats[cluster_id]
    print(f"Cluster {cluster_id:<3} {s['count']:<8} {s['mean_jumlah']:<12.0f} {s['min_jumlah']:<10.0f} "
          f"{s['max_jumlah']:<10.0f} Rp {s['mean_harga']:<13,.0f} Rp {s['min_harga']:<13,.0f} Rp {s['max_harga']:<13,.0f}")

In [ ]:
# ====================================================================
# TAHAP 10. INTERPRETASI DAN LABEL CLUSTER (FIXED)
# ====================================================================

# Urutkan cluster berdasarkan mean_jumlah (TERBESAR → TERKECIL)
sorted_clusters = sorted(cluster_stats.items(),
                         key=lambda x: x[1]['mean_jumlah'],
                         reverse=True)

cluster_label_map = {}
cluster_rank_map = {}

for rank, (cluster_id, stats) in enumerate(sorted_clusters):
    if rank == 0:
        label = "PEMESANAN TINGGI"
    elif rank == 1:
        label = "PEMESANAN SEDANG"
    else:
        label = "PEMESANAN RENDAH"

    cluster_label_map[cluster_id] = label
    cluster_rank_map[cluster_id] = rank

# Assign cluster_label_map to label_mapping for consistent use
label_mapping = cluster_label_map

# Tampilkan interpretasi
for cluster_id, stats in sorted_clusters:
    print(f"\nCluster {cluster_id}: {label_mapping[cluster_id]}")
    stats = cluster_stats[cluster_id]
    # Construct a descriptive string for each cluster
    if label_mapping[cluster_id] == "PEMESANAN TINGGI":
        description = f"Cluster dengan volume pemesanan tertinggi. Rata-rata {stats['mean_jumlah']:.0f} pcs per transaksi dengan harga rata-rata Rp {stats['mean_harga']:,.0f}."
    elif label_mapping[cluster_id] == "PEMESANAN SEDANG":
        description = f"Cluster dengan volume pemesanan menengah. Rata-rata {stats['mean_jumlah']:.0f} pcs per transaksi dengan harga rata-rata Rp {stats['mean_harga']:,.0f}."
    else:
        description = f"Cluster dengan volume pemesanan terendah. Rata-rata {stats['mean_jumlah']:.0f} pcs per transaksi dengan harga rata-rata Rp {stats['mean_harga']:,.0f}."

    print(f"  {description}")
    print(f"  Range pemesanan: {stats['min_jumlah']:.0f} - {stats['max_jumlah']:.0f} pcs")
    if 'std_jumlah' in stats: # Check if std_jumlah exists before printing
        print(f"  Standar deviasi: {stats['std_jumlah']:.0f} pcs")
    print(f"  Range harga: Rp {stats['min_harga']:,.0f} - Rp {stats['max_harga']:,.0f}")

In [ ]:
# ============================================================================
# TAHAP 11. MENGHITUNG SILHOUETTE SCORE AKHIR
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 11. SILHOUETTE SCORE AKHIR")
print("=" * 80)

silhouette_final = silhouette_score(X_normalized, df['cluster'])
silhouette_samples_final = silhouette_samples(X_normalized, df['cluster'])

print(f"\nSilhouette Score (K=3): {silhouette_final:.4f}")
print(f"Range Silhouette Score: -1 (buruk) hingga 1 (sempurna)")

if silhouette_final > 0.5:
    interpretasi = "BAIK - Cluster terpisah dengan jelas (well-separated)"
elif silhouette_final > 0.25:
    interpretasi = "CUKUP BAIK - Ada sedikit tumpang tindih antar cluster"
elif silhouette_final > 0:
    interpretasi = "KURANG BAIK - Cluster saling tumpang tindih"
else:
    interpretasi = "BURUK - Banyak data mungkin salah cluster"

print(f"Interpretasi: {interpretasi}")

# Silhouette per cluster
print(f"\nSilhouette Score per Cluster:")
for i in range(3):
    cluster_sil = silhouette_samples_final[df['cluster'] == i]
    print(f"  {label_mapping[i]} (Cluster {i}): {np.mean(cluster_sil):.4f}")

In [ ]:
# ============================================================================
# TAHAP 12. VISUALISASI HASIL CLUSTERING (SCATTER PLOT)
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 12. VISUALISASI HASIL CLUSTERING")
print("=" * 80)

# Warna untuk setiap cluster
warna_cluster = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# Buat figure dengan 3 subplot
fig = plt.figure(figsize=(18, 12))

# --- SUBPLOT 1: Scatter Plot Data Normalisasi ---
ax1 = fig.add_subplot(2, 2, 1)
for i, cluster_id in enumerate(sorted(df['cluster'].unique())):
    mask = df['cluster'] == cluster_id
    ax1.scatter(X_normalized[mask, 0], X_normalized[mask, 1],
               c=warna_cluster[i], label=f"{label_mapping[cluster_id]} (Cluster {cluster_id})",
               alpha=0.7, edgecolors='black', linewidth=0.5, s=80)

# Plot centroid
for i in range(3):
    ax1.scatter(centroids_normalized[i, 0], centroids_normalized[i, 1],
               c='black', marker='X', s=300, edgecolors='white', linewidth=2)

ax1.set_xlabel('Jumlah (Normalized)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Harga (Normalized)', fontsize=11, fontweight='bold')
ax1.set_title('Hasil Clustering K-Means (Data Normalisasi)', fontsize=12, fontweight='bold')
ax1.legend(loc='best', fontsize=8)
ax1.grid(alpha=0.3)

# --- SUBPLOT 2: Scatter Plot Data Asli ---
ax2 = fig.add_subplot(2, 2, 2)
for i, cluster_id in enumerate(sorted(df['cluster'].unique())):
    mask = df['cluster'] == cluster_id
    ax2.scatter(df.loc[mask, 'jumlah'], df.loc[mask, 'harga'],
               c=warna_cluster[i], label=f"{label_mapping[cluster_id]} (Cluster {cluster_id})",
               alpha=0.7, edgecolors='black', linewidth=0.5, s=80)

# Plot centroid dalam skala asli
for i in range(3):
    ax2.scatter(centroids_original[i, 0], centroids_original[i, 1],
               c='black', marker='X', s=300, edgecolors='white', linewidth=2)
    # Anotasi centroid
    ax2.annotate(f'Centroid {i}\n({centroids_original[i, 0]:.0f}, Rp {centroids_original[i, 1]:,.0f})',
                (centroids_original[i, 0], centroids_original[i, 1]),
                textcoords="offset points", xytext=(10, 10), fontsize=8,
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

ax2.set_xlabel('Jumlah Pemesanan (pcs)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Harga Satuan (Rp)', fontsize=11, fontweight='bold')
ax2.set_title('Hasil Clustering K-Means (Data Asli)', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=8)
ax2.grid(alpha=0.3)

# --- SUBPLOT 3: Boxplot Jumlah per Cluster ---
ax3 = fig.add_subplot(2, 2, 3)
cluster_order = sorted(df['cluster'].unique(),
                       key=lambda x: cluster_stats[x]['mean_jumlah'],
                       reverse=True)
data_box = [df[df['cluster'] == c]['jumlah'].values for c in cluster_order]
label_box = [f'{label_mapping[c]}\n(Cluster {c})' for c in cluster_order]

bp = ax3.boxplot(data_box, labels=label_box, patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], warna_cluster):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax3.set_title('Distribusi Jumlah Pemesanan per Cluster', fontsize=12, fontweight='bold')
ax3.set_ylabel('Jumlah Pemesanan (pcs)', fontsize=11)
ax3.grid(axis='y', alpha=0.3)

# Tambahkan anotasi statistik
for i, c in enumerate(cluster_order):
    s = cluster_stats[c]
    ax3.annotate(f"Mean: {s['mean_jumlah']:.0f}\nMin: {s['min_jumlah']:.0f}\nMax: {s['max_jumlah']:.0f}",
                xy=(i+1, s['max_jumlah']), fontsize=8, ha='center',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# --- SUBPLOT 4: Silhouette Plot ---
ax4 = fig.add_subplot(2, 2, 4)
y_lower = 10
for i in range(3):
    ith_cluster_sil = silhouette_samples_final[df['cluster'] == i]
    ith_cluster_sil.sort()
    size_cluster_i = len(ith_cluster_sil)
    y_upper = y_lower + size_cluster_i

    color = cm.nipy_spectral(float(i) / 3)
    ax4.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sil,
                      facecolor=color, edgecolor=color, alpha=0.7)
    ax4.text(-0.05, y_lower + 0.5 * size_cluster_i,
             f'{label_mapping[i]}', fontsize=9, fontweight='bold')
    y_lower = y_upper + 10

ax4.axvline(x=silhouette_final, color="red", linestyle="--", linewidth=2,
            label=f'Rata-rata: {silhouette_final:.4f}')
ax4.set_title('Silhouette Plot', fontsize=12, fontweight='bold')
ax4.set_xlabel('Koefisien Silhouette', fontsize=11)
ax4.set_ylabel('Cluster', fontsize=11)
ax4.set_yticks([])
ax4.set_xlim([-0.1, 1])
ax4.legend(loc='best')
ax4.grid(axis='x', alpha=0.3)

plt.suptitle('ANALISIS CLUSTERING K-MEANS PEMESANAN OPAK GAMBIR\nFitur: Jumlah & Harga | K = 3',
            fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('hasil_clustering_opak_gambir.png', dpi=150, bbox_inches='tight')
plt.show()

print("Visualisasi hasil clustering berhasil dibuat.")


In [ ]:
# ============================================================================
# TAHAP 13. TABEL RINGKASAN CLUSTER
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 13. TABEL RINGKASAN CLUSTER")
print("=" * 80)

print("\n" + "=" * 95)
print("TABEL RINGKASAN KARAKTERISTIK CLUSTER PEMESANAN OPAK GAMBIR")
print("=" * 95)
print(f"{'Nama Cluster':<22} {'Jumlah':<10} {'Rata² Jml':<12} {'Min Jml':<10} {'Max Jml':<10} {'Rata² Harga':<15} {'Range Harga':<20}")
print("-" * 95)

for cluster_id, stats in sorted_clusters:
    s = cluster_stats[cluster_id]
    range_harga = f"Rp {s['min_harga']:,.0f} - Rp {s['max_harga']:,.0f}"
    print(f"{label_mapping[cluster_id]:<22} {s['count']:<10} {s['mean_jumlah']:<12.0f} "
          f"{s['min_jumlah']:<10.0f} {s['max_jumlah']:<10.0f} "
          f"Rp {s['mean_harga']:<13,.0f} {range_harga:<20}")

print("=" * 95)

In [ ]:
# ============================================================================
# TAHAP 14. VISUALISASI TAMBAHAN: PIE CHART DISTRIBUSI
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 14. VISUALISASI DISTRIBUSI CLUSTER (PIE CHART)")
print("=" * 80)

fig, ax = plt.subplots(figsize=(8, 8))
sizes = [cluster_stats[c]['count'] for c in sorted(df['cluster'].unique())]
labels = [f'{label_mapping[c]}\n({sizes[i]} transaksi)' for i, c in enumerate(sorted(df['cluster'].unique()))]
colors_pie = ['#FF6B6B', '#4ECDC4', '#45B7D1']
explode = (0.05, 0.05, 0.05)

wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                                   autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
for autotext in autotexts:
    autotext.set_fontweight('bold')

ax.set_title('Distribusi Cluster Pemesanan Opak Gambir', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pie_chart_cluster.png', dpi=150, bbox_inches='tight')
plt.show()
print("Pie chart distribusi cluster berhasil dibuat.")


In [ ]:
# ============================================================================
# TAHAP 15. KESIMPULAN AKHIR
# ============================================================================
print("\n" + "=" * 80)
print("TAHAP 15. KESIMPULAN AKHIR")
print("=" * 80)

print("\n" + "=" * 80)
print("KESIMPULAN HASIL CLUSTERING K-MEANS")
print("PEMESANAN OPAK GAMBIR (FITUR: JUMLAH & HARGA, K=3)")
print("=" * 80)

print(f"""
1. KARAKTERISTIK MASING-MASING CLUSTER:
""")

for cluster_id, stats in sorted_clusters:
    print(f"   a. {cluster_label_map[cluster_id]} (Cluster {cluster_id}):")
    s = cluster_stats[cluster_id]
    print(f"      - Terdiri dari {s['count']} transaksi ({s['count']/len(df)*100:.1f}% dari total)")
    print(f"      - Volume pemesanan: {s['min_jumlah']:.0f} - {s['max_jumlah']:.0f} pcs per transaksi")
    print(f"      - Rata-rata pemesanan: {s['mean_jumlah']:.0f} pcs per transaksi")
    print(f"      - Harga rata-rata: Rp {s['mean_harga']:,.0f} per pcs")
    print(f"      - Range harga: Rp {s['min_harga']:,.0f} - Rp {s['max_harga']:,.0f}")
    print()

print(f"""
2. KATEGORI CLUSTER BERDASARKAN VOLUME PEMESANAN:
""")

sorted_by_mean = sorted(cluster_stats.items(), key=lambda x: x[1]['mean_jumlah'], reverse=True)
for rank, (cluster_id, s) in enumerate(sorted_by_mean):
    kategori = "Pemesanan Tinggi" if rank == 0 else "Pemesanan Sedang" if rank == 1 else "Pemesanan Rendah"
    print(f"   • {kategori}: Cluster {cluster_id} (Rata-rata {s['mean_jumlah']:.0f} pcs/transaksi)")

print(f"""

3. KUALITAS CLUSTER:
   - Silhouette Score: {silhouette_final:.4f}
   - Interpretasi: {interpretasi}
   - Cluster dengan silhouette tertinggi: {label_mapping[np.argmax([np.mean(silhouette_samples_final[df['cluster']==i]) for i in range(3)])]}

4. MANFAAT UNTUK PERENCANAAN PRODUKSI OPAK GAMBIR:

   a. Segmentasi Pelanggan:
      - {label_mapping[sorted_by_mean[0][0]]}: Pelanggan dengan volume pembelian besar
        → Perlu prioritas dalam layanan dan ketersediaan stok
      - {label_mapping[sorted_by_mean[1][0]]}: Pelanggan dengan volume pembelian menengah
        → Perlu program loyalitas untuk meningkatkan ke kategori tinggi
      - {label_mapping[sorted_by_mean[2][0]]}: Pelanggan dengan volume pembelian kecil
        → Perlu strategi promosi untuk meningkatkan volume pembelian

   b. Perencanaan Produksi:
      - Cluster {label_mapping[sorted_by_mean[0][0]]} ({sorted_by_mean[0][1]['count']} transaksi):
        Siapkan kapasitas produksi minimal {sorted_by_mean[0][1]['min_jumlah']:.0f} pcs
        dengan rata-rata {sorted_by_mean[0][1]['mean_jumlah']:.0f} pcs per pesanan
      - Cluster {label_mapping[sorted_by_mean[1][0]]} ({sorted_by_mean[1][1]['count']} transaksi):
        Siapkan kapasitas produksi {sorted_by_mean[1][1]['min_jumlah']:.0f} - {sorted_by_mean[1][1]['max_jumlah']:.0f} pcs
      - Cluster {label_mapping[sorted_by_mean[2][0]]} ({sorted_by_mean[2][1]['count']} transaksi):
        Siapkan kapasitas produksi hingga {sorted_by_mean[2][1]['max_jumlah']:.0f} pcs

   c. Strategi Harga:
      - Harga rata-rata bervariasi antar cluster:
        • {label_mapping[sorted_by_mean[0][0]]}: Rp {sorted_by_mean[0][1]['mean_harga']:,.0f}/pcs
        • {label_mapping[sorted_by_mean[1][0]]}: Rp {sorted_by_mean[1][1]['mean_harga']:,.0f}/pcs
        • {label_mapping[sorted_by_mean[2][0]]}: Rp {sorted_by_mean[2][1]['mean_harga']:,.0f}/pcs
      - Dapat digunakan untuk menentukan strategi diskon berdasarkan volume

   d. Manajemen Stok Bahan Baku:
      - Estimasi total produksi berdasarkan cluster:
        • {label_mapping[sorted_by_mean[0][0]]}: {sorted_by_mean[0][1]['mean_jumlah'] * sorted_by_mean[0][1]['count']:,.0f} pcs
        • {label_mapping[sorted_by_mean[1][0]]}: {sorted_by_mean[1][1]['mean_jumlah'] * sorted_by_mean[1][1]['count']:,.0f} pcs
        • {label_mapping[sorted_by_mean[2][0]]}: {sorted_by_mean[2][1]['mean_jumlah'] * sorted_by_mean[2][1]['count']:,.0f} pcs
      - Total estimasi kebutuhan: {sum(s['mean_jumlah'] * s['count'] for s in cluster_stats.values()):,.0f} pcs

5. REKOMENDASI:
   - Gunakan hasil clustering ini untuk membuat kebijakan produksi yang berbeda
     untuk setiap segmen pelanggan
   - Prioritaskan pemenuhan pesanan cluster PEMESANAN TINGGI
   - Kembangkan strategi untuk meningkatkan pelanggan dari cluster
     PEMESANAN RENDAH ke PEMESANAN SEDANG
   - Monitor pergerakan pelanggan antar cluster secara berkala
""")

print("=" * 80)
print("PROGRAM CLUSTERING K-MEANS SELESAI")
print("=" * 80)